In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from delta.table import DeltaTable

In [0]:
# Slowly Changing Dimension (SCD)
# SCD Type 1
# 1. Overwrite the existing value with new value
# 2. No history maintained
# 3. No need to maintain history table
# 4. No need to maintain version of table
# SCD Type 2 
# 1. Maintain history of table
# 2. Maintain version of table 
# SCD Type 3
# 1. Maintain history of table
# 2. Maintain version of table 

In [0]:
from delta.tables import DeltaTable

## SCD Type 1

In [0]:
%sql

create table sales(
  cust_id string,
  cust_name string,
  product string,
  price int
)

In [0]:
%sql
select * from sales

In [0]:
df=spark.createDataFrame([(101,'sunil','Electronics',784),(102,'anil','Hardware',1200),(103,'binod','Baby care',1000)],schema=['cust_id','cust_name','product','price'])


In [0]:
df.display()

In [0]:
df_sales = DeltaTable.forName(spark,'sales')
df_sales.toDF().display()

In [0]:
df_sales.alias('target').merge(df.alias('source'),"target.cust_id = source.cust_id").whenMatchedUpdate\
(set= {"target.cust_id":"source.cust_id","target.cust_name":"source.cust_name","target.product":"source.product","target.price":"source.price"}).whenNotMatchedInsertAll().execute()

In [0]:
df_sales.toDF().display()

In [0]:
df_sales.history().display()

## SCD Type 2

In [0]:
# SCD Type 2 
# 1. Maintain history of table(MAINTAIN OLD RECORD)
# 2. Maintain version of table (ALSO MAINTAIN NEW RECORD FOR SAME MATCHING ID)

In [0]:
df1=spark.createDataFrame([(101,'sunil','Electronics',784),(102,'anil','Hardware',18839),(103,'binod','Baby care',1000),(104,'surya','auto_mobile',1002)],schema=['cust_id','cust_name','product','price'])


In [0]:
df1.display()

In [0]:
df_sales=df_sales.toDF()

In [0]:
df_sales=df_sales.select("cust_id","cust_name","product","price",lit('Y').alias("cur_rec_ind"),current_timestamp()\
    .alias("start_ts"),current_timestamp().alias("end_ts"))

In [0]:
df_sales.display()

In [0]:
df_sales.write.mode("overwrite").option("mergeSchema","true").saveAsTable("sales")#save the latest data with changes table structure

In [0]:
spark.read.table("sales").display()

In [0]:
# df2=spark.createDataFrame([(101,'sunil','Electronics',784),(102,'anil','Hardware',18839),(103,'binod','Baby care',1000),(104,'surya','auto_mobile',1002),(105,'susil','mobile',10002)],schema=['cust_id','cust_name','product','price'])
# df2.display()

In [0]:
df1.createOrReplaceTempView("stg_vw1")

In [0]:
%sql

select * from stg_vw1

In [0]:
%sql

merge into sales trg
using stg_vw1 src
on trg.cust_id=src.cust_id
when matched then update set trg.cur_rec_ind='N', trg.end_ts=current_timestamp()
when not matched then insert (cust_id,cust_name,product,price,cur_rec_ind,start_ts,end_ts)
values(src.cust_id.src.cust_name,src.product,src.price,'Y',current_timestamp(),null)

In [0]:
%sql
merge into sales trg
using stg_vw1 src
on trg.cust_id=src.cust_id
when matched then update set trg.cur_rec_ind='N', trg.end_ts=current_timestamp()
when not matched then insert (cust_id, cust_name, product, price, cur_rec_ind, start_ts, end_ts)
values (src.cust_id, src.cust_name, src.product, src.price, 'Y', current_timestamp(), null)

In [0]:
%sql
select * from sales--doubdt cur_rec_id is update from 'Y' to 'N' but extra row is not inserted

In [0]:
spark.read.table("sales").display()

In [0]:
%sql

merge into sales trg
using stg_vw1 src
on trg.cust_id=src.cust_id and trg.cur_rec_ind='Y'
when matched and md5(concat_ws('-',src.cust_name,src.product,src.price)) != md5(concat_ws('-',trg.cust_name,trg.product,trg.price)) 
then update set trg.cur_rec_ind='N',trg.end_ts=current_timestamp()
when not matched then insert (cust_id, cust_name, product, price, cur_rec_ind, start_ts, end_ts)
values (src.cust_id, src.cust_name, src.product, src.price, 'Y', current_timestamp(), null)

In [0]:
%sql
select * from sales

In [0]:
%sql
with match_rec as (select stg_vw1.* from sales inner join stg_vw1 on sales.cust_id=stg_vw1.cust_id and sales.cur_rec_ind='N' and sales.end_ts is not null)
select * from match_rec

In [0]:
%sql
with match_rec as (select stg_vw1.* from stg_vw1 inner join sales on stg_vw1.cust_id=sales.cust_id 
and sales.cur_rec_ind='N' and sales.end_ts is not null )
  insert into sales (select *,'Y' as cur_rec_ind,current_timestamp() as start_ts,null as end_ts from match_rec)
  --select * from match_rec

In [0]:
%sql
select * from sales

## Optimize and Compaction

In [0]:
%sql

optimize sales zorder by (cust_id) --it will combain all the file into one file 

## Spark Config get and set

In [0]:
# spark.conf.set()
# spark.conf.set("spark.sql.legacy.allowCreatingManagedTableUsingNonemptyLocation","true")